# 2916. 子数组不同元素数目的平方和 II

[原文](https://leetcode.cn/problems/subarrays-distinct-element-sum-of-squares-ii/solution/2916xian-duan-shu-xian-bie-ji-zhao-bei-b-fj90/)

## 线段树

In [ ]:
# 对 [上次出现位置+1, r] 先查询旧和，再区间加一。
# 平方和增量为 2*旧和+区间长度。

In [ ]:
from typing import List


class LazySegmentTree:
    """区间加、区间求和；数组下标从 0 开始，区间均为闭区间。"""

    def __init__(self, nums: list[int]):
        self.n = len(nums)
        self.tree = [0] * (4 * self.n)
        self.lazy = [0] * (4 * self.n)
        self._build(1, 0, self.n - 1, nums)

    def _build(self, p, l, r, nums):
        if l == r:
            self.tree[p] = nums[l]
            return
        mid = (l + r) // 2
        self._build(p * 2, l, mid, nums)
        self._build(p * 2 + 1, mid + 1, r, nums)
        self.tree[p] = self.tree[p * 2] + self.tree[p * 2 + 1]

    def _apply(self, p, l, r, value):
        self.tree[p] += (r - l + 1) * value
        self.lazy[p] += value

    def _push(self, p, l, r):
        if self.lazy[p] != 0:
            mid = (l + r) // 2
            self._apply(p * 2, l, mid, self.lazy[p])
            self._apply(p * 2 + 1, mid + 1, r, self.lazy[p])
            self.lazy[p] = 0

    def add(self, left, right, value):
        self._add(1, 0, self.n - 1, left, right, value)

    def _add(self, p, l, r, ql, qr, value):
        if ql <= l and r <= qr:
            self._apply(p, l, r, value)
            return
        self._push(p, l, r)
        mid = (l + r) // 2
        if ql <= mid:
            self._add(p * 2, l, mid, ql, qr, value)
        if qr > mid:
            self._add(p * 2 + 1, mid + 1, r, ql, qr, value)
        self.tree[p] = self.tree[p * 2] + self.tree[p * 2 + 1]

    def query(self, left, right):
        return self._query(1, 0, self.n - 1, left, right)

    def _query(self, p, l, r, ql, qr):
        if ql <= l and r <= qr:
            return self.tree[p]
        self._push(p, l, r)
        mid = (l + r) // 2
        result = 0
        if ql <= mid:
            result += self._query(p * 2, l, mid, ql, qr)
        if qr > mid:
            result += self._query(p * 2 + 1, mid + 1, r, ql, qr)
        return result


class Solution:
    def sumCounts(self, nums: List[int]) -> int:
        mod = 10**9 + 7
        seg = LazySegmentTree([0] * len(nums))
        last = {}
        ending = 0  # 以上一个位置为右端点的所有子数组，其不同计数的平方和。
        answer = 0

        for r, x in enumerate(nums):
            prev = last.get(x, -1)
            left = prev + 1
            old_sum = seg.query(left, r)
            ending += 2 * old_sum + (r - left + 1)
            seg.add(left, r, 1)
            answer = (answer + ending) % mod
            last[x] = r

        return answer